# Alpha-Stable Distribution — Rolling & Point-in-Time Analysis

Fits stable distribution parameters to a rolling window of security returns using **libstable.py** — a Python port of `libstableR 1.0.2`.

The **stability index α** (alpha) is the key output:
| α value | Interpretation |
|---------|----------------|
| α = 2.0 | Gaussian (thin tails) |
| 1.5 < α < 2 | Moderately heavy tails |
| α = 1.0 | Cauchy (no finite mean) |
| α < 1.0 | Infinite variance and mean |

### MLE vs MLE2D — the key difference

| | `stable_fit_mle` | `stable_fit_mle2d` |
|---|---|---|
| **Optimised over** | All 4 params: (α, β, σ, μ) | Only 2 params: (α, β) |
| **σ, μ estimation** | Jointly from MLE | Analytically via McCulloch quantile tables at each step |
| **Accuracy** | Asymptotically most efficient | Slightly lower — σ/μ not jointly optimised |
| **Speed (N=252)** | ~270ms | ~85ms |
| **Best for** | Maximum precision, small N | Rolling windows, production |
| **Matches R** | `stable_fit_mle()` | `stable_fit_mle2d()` → calls `czab()` internally |

---
## 0. Configuration

In [ ]:
# ── Input ────────────────────────────────────────────────────────────────────
CSV_FILE      = "returns.csv"   # path to CSV (relative to notebook directory)
DATE_COL      = "date"          # date column name (case-insensitive)
RETURN_COL    = "return"        # return column name (case-insensitive)

# ── Rolling window ───────────────────────────────────────────────────────────
LOOKBACK      = 252             # rolling window in trading days

# ── Point-in-time date ───────────────────────────────────────────────────────
POINT_IN_TIME = None            # e.g. "2023-06-30"  →  None = last date

# ── Methods for POINT-IN-TIME (all fast — single window) ─────────────────────
PIT_MCCULLOCH    = True
PIT_KOUTROUVELIS = True
PIT_MLE2D        = True
PIT_MLE          = True

# ── Methods for ROLLING ───────────────────────────────────────────────────────
# JIT-compiled Nelder-Mead + 32-pt GL; approx. timings per 252-day window:
#   McCulloch: <1ms  |  Koutrouvelis: ~1ms  |  MLE2D: ~75ms  |  MLE: ~150ms
# For 2 000-row dataset (≈1 748 windows) with 8 cores → MLE2D ≈17s, MLE ≈33s
ROLL_MCCULLOCH    = True
ROLL_KOUTROUVELIS = True
ROLL_MLE2D        = True
ROLL_MLE          = True

# ── MLE tolerance for ROLLING (lower = faster but slightly less precise) ──────
ROLL_TOL      = 5e-3   # recommended; use 1e-3 for publication-quality

# ── Parallel rolling ─────────────────────────────────────────────────────────
PARALLEL      = True   # False = sequential (simpler debugging)
N_JOBS        = -1     # -1 = all cores
# ─────────────────────────────────────────────────────────────────────────────


---
## 1. Imports

In [ ]:
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

# libstable.py must be in the same directory as this notebook
sys.path.insert(0, os.path.dirname(os.path.abspath("libstable.py")))
import libstable as ls

print(f"libstable loaded  |  Numba JIT: {ls._NB_AVAILABLE}")
print(f"Lookback         : {LOOKBACK} days")
print(f"PIT methods      : McCulloch={PIT_MCCULLOCH}, Koutrouvelis={PIT_KOUTROUVELIS}, "
      f"MLE2D={PIT_MLE2D}, MLE={PIT_MLE}")
print(f"Rolling methods  : McCulloch={ROLL_MCCULLOCH}, Koutrouvelis={ROLL_KOUTROUVELIS}, "
      f"MLE2D={ROLL_MLE2D}, MLE={ROLL_MLE}")
print(f"Parallel rolling : {PARALLEL}  (N_JOBS={N_JOBS})")

---
## 2. Load data

In [ ]:
df = pd.read_csv(CSV_FILE)
df.columns = df.columns.str.strip().str.lower()

# Flexible column matching
date_cands   = [c for c in df.columns if "date" in c or c in ("time", "day")]
ret_cands    = [c for c in df.columns
                if any(k in c for k in ("return", "ret", "pnl"))
                and c not in date_cands]
dc = date_cands[0]   if date_cands   else df.columns[0]
rc = ret_cands[0]    if ret_cands    else df.columns[1]
if DATE_COL.lower()   in df.columns: dc = DATE_COL.lower()
if RETURN_COL.lower() in df.columns: rc = RETURN_COL.lower()

df = df[[dc, rc]].rename(columns={dc: "date", rc: "ret"})
df["date"] = pd.to_datetime(df["date"])
df["ret"]  = pd.to_numeric(df["ret"], errors="coerce")
df = df.dropna().sort_values("date").reset_index(drop=True)

# Auto-detect percentage vs decimal returns
if df["ret"].abs().median() > 1.0:
    print("Detected percentage returns — converting to decimal (/100)")
    df["ret"] /= 100.0

print(f"Loaded {len(df):,} rows  |  {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Return stats: mean={df['ret'].mean():.5f}  std={df['ret'].std():.5f}  "
      f"min={df['ret'].min():.4f}  max={df['ret'].max():.4f}")
df.tail(3)

---
## 3. Data overview

In [ ]:
from scipy import stats as sp_stats

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(df["date"], df["ret"], lw=0.6, color="steelblue", alpha=0.8)
axes[0].axhline(0, color="k", lw=0.4)
axes[0].set_title("Daily returns")
axes[0].set_ylabel("Return")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

r = df["ret"].values
axes[1].hist(r, bins=80, density=True, color="steelblue", alpha=0.6, label="Empirical")
xs_ = np.linspace(np.percentile(r,0.1), np.percentile(r,99.9), 300)
axes[1].plot(xs_, sp_stats.norm.pdf(xs_, r.mean(), r.std()), "r-", lw=2, label="Normal")
axes[1].set_title("Return distribution")
axes[1].legend()

sp_stats.probplot(r, dist="norm", plot=axes[2])
axes[2].set_title("Normal Q-Q plot")
axes[2].get_lines()[1].set_color("red")

plt.tight_layout()
plt.show()

kurt = pd.Series(r).kurt()
print(f"Excess kurtosis: {kurt:.2f}  "
      f"({'heavy tails — stable α likely < 2' if kurt > 1 else 'near-Gaussian'})")

---
## 4. Helper — fit all methods on one window

In [ ]:
def fit_stable(returns,
               run_mcc=True, run_kou=True, run_mle2d=True, run_mle=True,
               tol=1e-3):
    """
    Fit stable distribution to `returns` using up to 4 methods.

    Parameters
    ----------
    tol : float
        Nelder-Mead convergence tolerance for MLE/MLE2D.
        Use 1e-3 (default) for point-in-time; 5e-3 for fast rolling.

    Returns dict  key → {'alpha','beta','sigma','mu','time_ms'}  or None if skipped.
    Keys: 'mcc', 'kou', 'mle2d', 'mle'
    """
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 10:
        return {k: None for k in ("mcc","kou","mle2d","mle")}

    def _pack(pars, t0):
        return {"alpha": float(pars[0]), "beta": float(pars[1]),
                "sigma": float(pars[2]), "mu":   float(pars[3]),
                "time_ms": (time.perf_counter()-t0)*1000}

    out = {}

    # McCulloch (always computed — used to seed other methods)
    t0 = time.perf_counter()
    p_init = ls.stable_fit_init(r)
    out["mcc"] = _pack(p_init, t0) if run_mcc else None

    p_seed = p_init.tolist()

    if run_kou:
        t0 = time.perf_counter()
        try:    out["kou"] = _pack(ls.stable_fit_koutrouvelis(r, pars_init=p_seed), t0)
        except: out["kou"] = None
    else:
        out["kou"] = None

    if run_mle2d:
        t0 = time.perf_counter()
        try:    out["mle2d"] = _pack(ls.stable_fit_mle2d(r, pars_init=p_seed, tol=tol), t0)
        except: out["mle2d"] = None
    else:
        out["mle2d"] = None

    if run_mle:
        t0 = time.perf_counter()
        try:    out["mle"] = _pack(ls.stable_fit_mle(r, pars_init=p_seed, tol=tol), t0)
        except: out["mle"] = None
    else:
        out["mle"] = None

    return out


METHOD_LABELS = {"mcc":"McCulloch", "kou":"Koutrouvelis",
                 "mle2d":"MLE2D",    "mle":"MLE (full)"}
METHOD_COLORS = {"mcc":"#2ecc71",   "kou":"#3498db",
                 "mle2d":"#e67e22",  "mle":"#e74c3c"}


def results_table(res, title=""):
    rows = []
    for k, lbl in METHOD_LABELS.items():
        v = res.get(k)
        if v is None: continue
        rows.append({"Method": lbl,
                     "α (alpha)": f"{v['alpha']:.4f}",
                     "β (beta)": f"{v['beta']:+.4f}",
                     "σ (sigma)": f"{v['sigma']:.6f}",
                     "μ (mu)": f"{v['mu']:+.6f}",
                     "Time (ms)": f"{v['time_ms']:.1f}"})
    tbl = pd.DataFrame(rows).set_index("Method")
    if title:
        print(f"\n{'─'*58}\n  {title}\n{'─'*58}")
    return tbl

print("Helpers defined.")


---
## 5. Point-in-time analysis

Fit stable distribution to the **`LOOKBACK`-day window ending at `POINT_IN_TIME`** using all enabled methods.

In [ ]:
# Resolve target date
if POINT_IN_TIME is None:
    pit_date = df["date"].iloc[-1]
else:
    pit_date = df.loc[df["date"] <= pd.Timestamp(POINT_IN_TIME), "date"].iloc[-1]

pit_idx   = df.index[df["date"] == pit_date][0]
start_idx = max(0, pit_idx - LOOKBACK + 1)
pit_rets  = df.loc[start_idx:pit_idx, "ret"].values

print(f"Point-in-time date : {pit_date.date()}")
print(f"Window             : {df['date'].iloc[start_idx].date()} → "
      f"{pit_date.date()}  ({len(pit_rets)} days)")

t0 = time.perf_counter()
pit_res = fit_stable(pit_rets,
                     run_mcc=PIT_MCCULLOCH,  run_kou=PIT_KOUTROUVELIS,
                     run_mle2d=PIT_MLE2D,    run_mle=PIT_MLE)
print(f"Total fit time     : {(time.perf_counter()-t0)*1000:.0f}ms")

tbl = results_table(pit_res, f"Point-in-time: {pit_date.date()}  [{LOOKBACK}-day window]")
display(tbl)

In [ ]:
# Fitted PDFs vs empirical histogram
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x_lo, x_hi = np.percentile(pit_rets, 0.5), np.percentile(pit_rets, 99.5)
x_plot = np.linspace(x_lo, x_hi, 500)

for ax, yscale in zip(axes, ["linear", "log"]):
    ax.hist(pit_rets, bins=50, density=True,
            color="steelblue", alpha=0.35, label=f"Empirical ({len(pit_rets)}d)")

    for k, lbl in METHOD_LABELS.items():
        v = pit_res.get(k)
        if v is None: continue
        try:
            pdf_ = ls.stable_pdf(x_plot, [v["alpha"],v["beta"],v["sigma"],v["mu"]])
            ax.plot(x_plot, pdf_, color=METHOD_COLORS[k], lw=2,
                    label=f"{lbl}  α={v['alpha']:.3f}")
        except Exception:
            pass

    # Normal overlay
    ax.plot(x_plot, sp_stats.norm.pdf(x_plot, pit_rets.mean(), pit_rets.std()),
            "k--", lw=1.5, alpha=0.5, label="Normal")

    ax.set_xlabel("Daily return")
    ax.set_ylabel("Density")
    ax.set_title(f"Fitted PDFs — {pit_date.date()}  ({'linear' if yscale=='linear' else 'log scale'})")
    ax.legend(fontsize=9)
    if yscale == "log":
        ax.set_yscale("log")
        ax.set_ylim(bottom=1e-3)

plt.tight_layout()
plt.show()

---
## 6. Rolling computation

Each row `i ≥ LOOKBACK-1` uses the window `[i-LOOKBACK+1 … i]`.

In [ ]:
ret_arr    = df["ret"].values
n          = len(ret_arr)
n_windows  = n - LOOKBACK + 1

roll_flags = dict(run_mcc=ROLL_MCCULLOCH, run_kou=ROLL_KOUTROUVELIS,
                  run_mle2d=ROLL_MLE2D,   run_mle=ROLL_MLE)

# ── Estimate per-window time (uses ROLL_TOL for realistic timing) ─────────────
_ = fit_stable(ret_arr[:LOOKBACK], **roll_flags, tol=ROLL_TOL)  # warm up
t0 = time.perf_counter()
for _ in range(3):
    fit_stable(ret_arr[:LOOKBACK], **roll_flags, tol=ROLL_TOL)
ms_per_window = (time.perf_counter()-t0)/3 * 1000
est_s = ms_per_window * n_windows / 1000

active_methods = [k for k,v in roll_flags.items()
                  if v and k.replace("run_","") in METHOD_LABELS]
active_keys    = [k.replace("run_","") for k in active_methods]

n_cpu_avail = __import__('multiprocessing').cpu_count()
est_par_s   = est_s / n_cpu_avail if PARALLEL else est_s

print(f"{n_windows:,} windows  |  {ms_per_window:.0f}ms/window (tol={ROLL_TOL})")
print(f"Methods : {[METHOD_LABELS[k] for k in active_keys]}")
print(f"Seq est : {est_s:.0f}s ({est_s/60:.1f} min)")
print(f"Par est : {est_par_s:.0f}s on {n_cpu_avail} cores  "
      f"({'enabled' if PARALLEL else 'PARALLEL=False'})") 


In [ ]:
# ── Worker for parallel/sequential execution ─────────────────────────────────
def _roll_worker(args):
    """
    Thread-safe worker.  Threads share the main process's already-imported
    libstable module; Numba JIT releases the GIL so threads run in parallel.
    """
    import numpy as _np
    import time as _time
    import libstable as _ls   # instant in threads — returns cached sys.modules entry

    end_idx, window, flags, tol = args
    r = _np.asarray(window, dtype=float)
    r = r[_np.isfinite(r)]
    if len(r) < 10:
        return end_idx, {k: None for k in ("mcc", "kou", "mle2d", "mle")}

    def _pack(p, t0):
        return {"alpha": float(p[0]), "beta": float(p[1]),
                "sigma": float(p[2]), "mu":   float(p[3]),
                "time_ms": (_time.perf_counter() - t0) * 1000}

    out = {}
    t0 = _time.perf_counter()
    p_init = _ls.stable_fit_init(r)
    out["mcc"] = _pack(p_init, t0) if flags["mcc"] else None
    p_seed = p_init.tolist()

    if flags["kou"]:
        t0 = _time.perf_counter()
        try:    out["kou"] = _pack(_ls.stable_fit_koutrouvelis(r, pars_init=p_seed), t0)
        except: out["kou"] = None
    else: out["kou"] = None

    if flags["mle2d"]:
        t0 = _time.perf_counter()
        try:    out["mle2d"] = _pack(_ls.stable_fit_mle2d(r, pars_init=p_seed, tol=tol), t0)
        except: out["mle2d"] = None
    else: out["mle2d"] = None

    if flags["mle"]:
        t0 = _time.perf_counter()
        try:    out["mle"] = _pack(_ls.stable_fit_mle(r, pars_init=p_seed, tol=tol), t0)
        except: out["mle"] = None
    else: out["mle"] = None

    return end_idx, out

print("Worker function defined.")


In [ ]:
# ── Pre-allocate result storage ───────────────────────────────────────────────
param_names  = ["alpha", "beta", "sigma", "mu"]
rolling_data = {k: {p: np.full(n, np.nan) for p in param_names}
                for k in active_keys}

flags_dict = {k.replace("run_",""): v for k, v in roll_flags.items()}

t_start = time.perf_counter()

if PARALLEL:
    # ── Parallel via fork-based Pool ──────────────────────────────────────────
    # 'fork' (not 'spawn') is used so child processes inherit the parent's
    # already-compiled Numba JIT cache — no per-worker recompilation, no
    # spawn/pickle deadlock that would hang the notebook on macOS.
    import multiprocessing as mp
    import warnings
    warnings.filterwarnings("ignore")  # suppress any fork-after-thread advisory

    n_cpu = mp.cpu_count() if N_JOBS == -1 else N_JOBS
    n_cpu = min(n_cpu, n_windows)
    print(f"Running parallel on {n_cpu} processes (tol={ROLL_TOL}) ...")

    tasks = [(end_idx,
              ret_arr[end_idx - LOOKBACK + 1 : end_idx + 1].tolist(),
              flags_dict,
              ROLL_TOL)
             for end_idx in range(LOOKBACK - 1, n)]

    ctx = mp.get_context("fork")   # inherits Numba cache → real parallelism
    with ctx.Pool(n_cpu) as pool:
        results_list = pool.map(_roll_worker, tasks,
                                chunksize=max(1, len(tasks)//n_cpu//4))

    for end_idx, res in results_list:
        for k in active_keys:
            v = res.get(k)
            if v is not None:
                for p in param_names:
                    rolling_data[k][p][end_idx] = v[p]

else:
    # ── Sequential with progress bar ─────────────────────────────────────────
    try:
        from tqdm.auto import tqdm
        iterator = tqdm(range(LOOKBACK - 1, n), desc="Rolling")
    except ImportError:
        iterator = range(LOOKBACK - 1, n)
        print("(install tqdm for progress bar: pip install tqdm)")

    for end_idx in iterator:
        window = ret_arr[end_idx - LOOKBACK + 1 : end_idx + 1]
        res = fit_stable(window, **roll_flags, tol=ROLL_TOL)
        for k in active_keys:
            v = res.get(k)
            if v is not None:
                for p in param_names:
                    rolling_data[k][p][end_idx] = v[p]

elapsed = time.perf_counter() - t_start
print(f"\nDone in {elapsed:.1f}s  ({elapsed/n_windows*1000:.0f}ms/window)")


In [ ]:
# Build tidy DataFrame
roll_df = pd.DataFrame({"date": df["date"]})
for k in active_keys:
    for p in param_names:
        roll_df[f"{METHOD_LABELS[k]}_{p}"] = rolling_data[k][p]

roll_df = roll_df.iloc[LOOKBACK - 1:].reset_index(drop=True)
print(f"Rolling DataFrame: {len(roll_df):,} rows × {len(roll_df.columns)} columns")
roll_df.tail(3)

---
## 7. Rolling α — all methods

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})

# ── Rolling alpha ─────────────────────────────────────────────────────────────
ax = axes[0]
for k in active_keys:
    col = f"{METHOD_LABELS[k]}_alpha"
    if col in roll_df.columns:
        ax.plot(roll_df["date"], roll_df[col],
                color=METHOD_COLORS[k], lw=1.4, alpha=0.85,
                label=f"{METHOD_LABELS[k]}  (median={roll_df[col].median():.3f})")

ax.axhline(2.0, color="#7f8c8d", lw=1.0, ls="--", alpha=0.5, label="α=2 (Gaussian)")
ax.axhline(1.0, color="#7f8c8d", lw=0.8, ls=":",  alpha=0.4, label="α=1 (Cauchy)")
ax.axvline(pit_date, color="#8e44ad", lw=1.5, ls="-", alpha=0.8,
           label=f"PIT: {pit_date.date()}")
ax.set_ylabel(f"Stability index α  ({LOOKBACK}-day rolling)", fontsize=12)
ax.set_ylim(0.3, 2.15)
ax.set_title(f"Rolling {LOOKBACK}-day stable α", fontsize=14)
ax.legend(loc="lower left", ncol=3, fontsize=9)
ax.grid(True, alpha=0.25)

# ── Return series ─────────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.fill_between(df["date"], df["ret"], 0,
                 where=df["ret"] >= 0, color="#27ae60", alpha=0.7, linewidth=0)
ax2.fill_between(df["date"], df["ret"], 0,
                 where=df["ret"] <  0, color="#e74c3c", alpha=0.7, linewidth=0)
ax2.axhline(0, color="k", lw=0.5)
ax2.axvline(pit_date, color="#8e44ad", lw=1.5, ls="-", alpha=0.8)
ax2.set_ylabel("Daily return", fontsize=10)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

---
## 8. All four stable parameters (rolling)

In [ ]:
param_meta = [
    ("alpha", "α  stability index", (0.3, 2.1)),
    ("beta",  "β  skewness",         (-1.05, 1.05)),
    ("sigma", "σ  scale",            None),
    ("mu",    "μ  location (S0)",    None),
]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

for ax, (p, ylabel, ylim) in zip(axes, param_meta):
    for k in active_keys:
        col = f"{METHOD_LABELS[k]}_{p}"
        if col in roll_df.columns:
            ax.plot(roll_df["date"], roll_df[col],
                    color=METHOD_COLORS[k], lw=1.0, alpha=0.8,
                    label=METHOD_LABELS[k])
    if p == "alpha":
        ax.axhline(2.0, color="#7f8c8d", lw=0.8, ls="--", alpha=0.5)
        ax.axhline(1.0, color="#7f8c8d", lw=0.8, ls=":",  alpha=0.4)
    if p == "beta":
        ax.axhline(0, color="#7f8c8d", lw=0.5, ls="-", alpha=0.3)
    ax.axvline(pit_date, color="#8e44ad", lw=1.0, ls="-", alpha=0.6)
    if ylim:
        ax.set_ylim(ylim)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(loc="upper right", ncol=4, fontsize=8)
    ax.grid(True, alpha=0.2)

axes[0].set_title(f"Rolling {LOOKBACK}-day stable parameters — all methods", fontsize=13)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

---
## 9. Method comparison — Δα between methods

In [ ]:
if len(active_keys) >= 2:
    pairs = [(active_keys[i], active_keys[j])
             for i in range(len(active_keys))
             for j in range(i+1, len(active_keys))]

    fig, axes = plt.subplots(len(pairs), 1,
                              figsize=(16, 3.8*len(pairs)), sharex=True)
    if len(pairs) == 1: axes = [axes]

    for ax, (ka, kb) in zip(axes, pairs):
        ca = f"{METHOD_LABELS[ka]}_alpha"
        cb = f"{METHOD_LABELS[kb]}_alpha"
        if ca not in roll_df.columns or cb not in roll_df.columns:
            continue
        d = roll_df[ca] - roll_df[cb]
        ax.plot(roll_df["date"], d, lw=0.9, color="#8e44ad", alpha=0.8)
        ax.fill_between(roll_df["date"], d, 0, alpha=0.15, color="#8e44ad")
        ax.axhline(0, color="k", lw=0.5)
        ax.axvline(pit_date, color="#8e44ad", lw=1.0, ls="--", alpha=0.6)
        ax.set_ylabel(f"Δα", fontsize=10)
        ax.set_title(f"{METHOD_LABELS[ka]} − {METHOD_LABELS[kb]}:  "
                     f"mean={d.mean():.4f}   std={d.std():.4f}   "
                     f"max|Δ|={d.abs().max():.4f}")
        ax.grid(True, alpha=0.2)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.suptitle("Rolling α — pairwise method differences", fontsize=13, y=1.005)
    plt.tight_layout()
    plt.show()
else:
    print("Enable ≥ 2 methods to see comparisons.")

---
## 10. Annual box-plots of rolling α

In [ ]:
# Choose best available method for the box-plot
best_key = next((k for k in ["mle","mle2d","kou","mcc"] if k in active_keys), None)

if best_key:
    col = f"{METHOD_LABELS[best_key]}_alpha"
    tmp = roll_df[["date", col]].dropna().copy()
    tmp["year"] = tmp["date"].dt.year
    years = sorted(tmp["year"].unique())
    by_year = [tmp.loc[tmp["year"]==y, col].values for y in years]

    fig, ax = plt.subplots(figsize=(max(8, len(years)), 5))
    bp = ax.boxplot(by_year, labels=years, patch_artist=True,
                    medianprops=dict(color="red", lw=2),
                    boxprops=dict(facecolor="#aed6f1", alpha=0.7))
    ax.axhline(2.0, color="gray", lw=1, ls="--", alpha=0.5, label="α=2 (Gaussian)")
    ax.axhline(1.0, color="gray", lw=0.8, ls=":",  alpha=0.4, label="α=1 (Cauchy)")
    ax.set_xlabel("Year"); ax.set_ylabel("Rolling α")
    ax.set_title(f"Annual distribution of {LOOKBACK}-day rolling α  [{METHOD_LABELS[best_key]}]")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

---
## 11. Rolling summary statistics

In [ ]:
rows = []
for k in active_keys:
    for p in param_names:
        col = f"{METHOD_LABELS[k]}_{p}"
        if col not in roll_df.columns: continue
        s = roll_df[col].dropna()
        rows.append({"Method": METHOD_LABELS[k], "Param": p,
                     "Mean": f"{s.mean():.4f}", "Std": f"{s.std():.4f}",
                     "Min":  f"{s.min():.4f}",  "p25": f"{s.quantile(0.25):.4f}",
                     "Med":  f"{s.median():.4f}","p75": f"{s.quantile(0.75):.4f}",
                     "Max":  f"{s.max():.4f}"})

smry = pd.DataFrame(rows).set_index(["Method","Param"])

print("α summary (all methods):")
display(smry.xs("alpha", level="Param"))
print("\nAll parameters:")
display(smry)

---
## 12. Export

In [ ]:
roll_path = "rolling_stable_params.csv"
roll_df.to_csv(roll_path, index=False)
print(f"Rolling results → {roll_path}  ({len(roll_df):,} rows)")

pit_path = "pit_stable_params.csv"
tbl.to_csv(pit_path)
print(f"Point-in-time   → {pit_path}")

print(f"\nColumns in rolling output:")
for c in roll_df.columns:
    print(f"  {c}")

---
## Appendix A — MLE vs MLE2D: implementation detail

```
stable_fit_mle  (full 4D MLE)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Reparametrised search space (unconstrained Nelder-Mead):
   α → tan(π/2·(α−1))    β → tan(π/2·β)
   σ → log(σ)             μ → μ
 At each evaluation: compute PDF(xᵢ|α,β,σ,μ) for all i
 Cost: −Σ log PDF(xᵢ)  over all four parameters jointly
 Matches R's  stable_fit_mle()  → stable_fit_iter_whole()

stable_fit_mle2d  (2D MLE)  ← matches R's stable_fit_mle2d()
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Search space: only (α, β) via Nelder-Mead
 At each evaluation:
   1. σ, μ  ←  czab(α, β, IQR, median)   [McCulloch tables]
   2. cost   =  −Σ log PDF(xᵢ | α, β, σ̂, μ̂)
 σ/μ are PINNED by McCulloch, not jointly optimised
 Matches R's  stable_fit_mle2d()  → stable_fit_iter()

Speed comparison (N=252, warm Numba):
  MLE   : ~270ms   (4 params × PDF calls per Nelder-Mead step)
  MLE2D : ~85ms    (2 params × same PDF calls, faster convergence)
  ➜ For rolling 252d × 1750 windows:
     MLE   → ~470s  (8 min)
     MLE2D → ~150s  (2.5 min)
     MLE2D ∥ 8 cores → ~20s
```

---
## Appendix B — Generate synthetic test CSV

In [ ]:
# Run this cell once to create a synthetic returns.csv with two regimes.
# Regime 1 (2016-2019): α=1.7, near-normal
# Regime 2 (2020-2023): α=1.4, heavier tails (stress period)

import pandas as pd, numpy as np
import libstable as ls

np.random.seed(42)
dates = pd.bdate_range("2016-01-04", periods=2000)
r1 = ls.stable_rnd(1000, [1.7,  0.0,  0.008,  0.0003], seed=1)
r2 = ls.stable_rnd(1000, [1.4, -0.1,  0.012, -0.0002], seed=2)

out = pd.DataFrame({"date": dates, "return": np.concatenate([r1, r2])})
out.to_csv("returns.csv", index=False)
print(f"Saved returns.csv  ({len(out)} rows)")
print(f"Period: {out.date.min().date()} → {out.date.max().date()}")
out.head(3)